## Imports

In [1]:
import math
import os
import time
import requests
import pandas as pd
import json
import numpy as np
from pathlib import Path
from typing import Optional, Dict, List, Union
from unidecode import unidecode

import pandas_gbq
from google.auth import default
from google.cloud import bigquery
from google.api_core.exceptions import NotFound

/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/google/api_core/_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please u

In [2]:
import warnings
warnings.filterwarnings("ignore")

## Diretório

In [3]:
BASE_DIR = Path("data")
RAW_DIR = BASE_DIR / "raw"
TRUSTED_DIR = BASE_DIR / "trusted"
ANALYTICS_DIR = BASE_DIR / "analytics"

for path in [RAW_DIR, TRUSTED_DIR, ANALYTICS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

## Google BigQuery

In [4]:
project_id = 'loft-dl-fintech'

## Bases

In [5]:
'''
WITH 
SIMULATIONS AS (
  SELECT *
  FROM `loft-dl-fintech.cp_gold.credit_fact` AS CF
  WHERE requested_at >= '2026-05-01'
),
MODEL_DATA AS (
  SELECT 
    CAST(REGEXP_REPLACE(documento_novo, r'[^0-9]', '') AS INT64) AS CPF_CNPJ,
    CAST(id_externo AS INT64) AS contract_id,
    DATE(data) AS requested_at,
    FORMAT_DATE('%Y-%m', DATE(data)) AS request_month,
    * 
  FROM `loft-dl-fintech.bronze_credpago.consulta_realizada` AS CR
  WHERE DATE(data) >= '2026-05-01'
),
MODEL_DATA_UP AS (
  SELECT *
  FROM MODEL_DATA
  QUALIFY ROW_NUMBER() OVER (
      PARTITION BY contract_id
      ORDER BY data DESC
  ) = 1
),
SIMULATIONS_UP AS (
  SELECT *
  FROM SIMULATIONS
  QUALIFY ROW_NUMBER() OVER (
      PARTITION BY contract_id
      ORDER BY requested_at DESC
  ) = 1
)
SELECT *
FROM SIMULATIONS_UP AS S LEFT JOIN MODEL_DATA_UP AS M
ON S.contract_id = M.contract_id
'''

"\nWITH \nSIMULATIONS AS (\n  SELECT *\n  FROM `loft-dl-fintech.cp_gold.credit_fact` AS CF\n  WHERE requested_at >= '2026-05-01'\n),\nMODEL_DATA AS (\n  SELECT \n    CAST(REGEXP_REPLACE(documento_novo, r'[^0-9]', '') AS INT64) AS CPF_CNPJ,\n    CAST(id_externo AS INT64) AS contract_id,\n    DATE(data) AS requested_at,\n    FORMAT_DATE('%Y-%m', DATE(data)) AS request_month,\n    * \n  FROM `loft-dl-fintech.bronze_credpago.consulta_realizada` AS CR\n  WHERE DATE(data) >= '2026-05-01'\n),\nMODEL_DATA_UP AS (\n  SELECT *\n  FROM MODEL_DATA\n  QUALIFY ROW_NUMBER() OVER (\n      PARTITION BY contract_id\n      ORDER BY data DESC\n  ) = 1\n),\nSIMULATIONS_UP AS (\n  SELECT *\n  FROM SIMULATIONS\n  QUALIFY ROW_NUMBER() OVER (\n      PARTITION BY contract_id\n      ORDER BY requested_at DESC\n  ) = 1\n)\nSELECT *\nFROM SIMULATIONS_UP AS S LEFT JOIN MODEL_DATA_UP AS M\nON S.contract_id = M.contract_id\n"

### CredPago - Consulta Realizada

In [6]:
query_credpago = '''
WITH
MODEL_DATA AS (
  SELECT 
    CAST(REGEXP_REPLACE(documento_novo, r'[^0-9]', '') AS INT64) AS CPF_CNPJ,
    CAST(id_externo AS INT64) AS contract_id,
    DATE(data) AS requested_at,
    FORMAT_DATE('%Y-%m', DATE(data)) AS request_month,
    * 
  FROM `loft-dl-fintech.bronze_credpago.consulta_realizada` AS CR
  WHERE DATE(data) >= '2026-05-01'
)
SELECT *
FROM MODEL_DATA
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY contract_id
    ORDER BY requested_at DESC
) = 1
'''
credpago_df = pd.read_gbq(query_credpago, project_id=project_id)
credpago_df

,CPF_CNPJ,contract_id,requested_at,request_month,id_consulta,limite_mensal,result_pre_analise,renda_considerada,id_bureau,risco_cartoes,...,_sdc_batched_at,faixa_renda,data,score_imobiliaria,score_hvar,decisao_bureau,score_bvs,id_funcionalidade,score_hva3,json_retornado
0,70807521248,4016844,2026-05-08,2026-05,5769969,0E-9,APROVAR,1986.500000000,1,0,...,2026-05-08 17:01:55.389000+00:00,0,2026-05-08 12:37:45+00:00,C,0,BLEND3_3,583,33,<NA>,"{""success"":true,""message"":{""manual"":false,""nod..."
1,1390353036,4050748,2026-05-25,2026-05,5844344,0E-9,APROVAR,2603.000000000,1,0,...,2026-05-25 15:05:28.717000+00:00,0,2026-05-25 11:38:10+00:00,NI,0,BLEND3_3,506,34,<NA>,"{""success"":true,""message"":{""pessoas"":[{""nome"":..."
2,7114053525,4052776,2026-05-14,2026-05,5794702,0E-9,APROVAR,7261.000000000,1,0,...,2026-05-14 15:08:39.698000+00:00,0,2026-05-14 10:10:45+00:00,NI,0,BLEND3_3,792,32,<NA>,"{""success"":true,""message"":{""manual"":false,""nod..."
3,43119138851,4068320,2026-05-06,2026-05,5756141,0E-9,APROVAR,4452.500000000,1,0,...,2026-05-06 15:14:20.461000+00:00,0,2026-05-06 10:58:01+00:00,C,0,BLEND3_3,796,32,<NA>,"{""success"":true,""message"":{""manual"":false,""nod..."
4,1152804995,4069507,2026-05-04,2026-05,5746287,0E-9,APROVAR,3219.500000000,1,0,...,2026-05-04 21:10:08.470000+00:00,0,2026-05-04 17:25:58+00:00,B,0,BLEND3_3,581,34,<NA>,"{""success"":true,""message"":{""manual"":false,""nod..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86492,3814735030,4173058,2026-05-25,2026-05,5848917,0E-9,APROVAR,4178.500000000,1,0,...,2026-05-25 21:05:54.970000+00:00,0,2026-05-25 17:49:04+00:00,E,0,BLEND3_3,674,34,<NA>,"{""success"":true,""message"":{""pessoas"":[{""nome"":..."
86493,1471724972,4173104,2026-05-25,2026-05,5848972,0E-9,APROVAR,1849.500000000,1,0,...,2026-05-25 21:05:54.992000+00:00,0,2026-05-25 17:54:16+00:00,NI,0,BLEND3_3,439,33,<NA>,"{""success"":true,""message"":{""pessoas"":[{""nome"":..."
86494,11225532930,4173137,2026-05-25,2026-05,5849010,0E-9,APROVAR,4726.500000000,1,0,...,2026-05-25 21:05:55.007000+00:00,0,2026-05-25 17:59:28+00:00,C,0,BLEND3_3,491,34,<NA>,"{""success"":true,""message"":{""pessoas"":[{""nome"":..."
86495,70383410142,4173377,2026-05-25,2026-05,5849310,0E-9,APROVAR,1986.500000000,1,0,...,2026-05-26 09:08:36.109000+00:00,0,2026-05-25 21:34:59+00:00,B,0,BLEND3_3,524,33,<NA>,"{""success"":true,""message"":{""pessoas"":[{""nome"":..."


In [7]:
credpago_df.columns

Index(['CPF_CNPJ', 'contract_id', 'requested_at', 'request_month',
       'id_consulta', 'limite_mensal', 'result_pre_analise',
       'renda_considerada', 'id_bureau', 'risco_cartoes', 'id_pessoa',
       'protestados_outros_estados', 'id_externo', 'renda_mensal',
       '_sdc_table_version', 'ocorrencia_debito', 'score_bureau',
       'descricao_probabilidade_inadimplencia', 'motor', 'protesto_sao_paulo',
       'probabilidade_inadimplencia', 'score_serasa', '_sdc_received_at',
       '_sdc_sequence', 'score_medio_4kst', 'regua', 'documento', 'letra',
       'idade', 'documento_novo', '_sdc_batched_at', 'faixa_renda', 'data',
       'score_imobiliaria', 'score_hvar', 'decisao_bureau', 'score_bvs',
       'id_funcionalidade', 'score_hva3', 'json_retornado'],
      dtype='object')

In [8]:
credpago_df['decisao_bureau'].value_counts(dropna=False)

decisao_bureau
BLEND3_3                74402
BLEND_REGRESSAO_2026     7291
HVA3                     3130
BVS_CUSTOM               1317
HVA4                      338
                           17
None                        2
Name: count, dtype: int64

In [9]:
credpago_df['decisao_bureau'].value_counts(normalize=True, dropna=False)

decisao_bureau
BLEND3_3                0.860169
BLEND_REGRESSAO_2026    0.084292
HVA3                    0.036186
BVS_CUSTOM              0.015226
HVA4                    0.003908
                        0.000197
None                    0.000023
Name: proportion, dtype: float64

In [10]:
credpago_df[credpago_df['decisao_bureau'] == 'BLEND3_3']['json_retornado'].values[-1]

'{"success":true,"message":{"pessoas":[{"nome":"ROSELI ROCHA","documento":"12536359867","classificacao":"C","tipoPessoa":"FISICA","idade":59,"scores":{"BVS_CUSTOM":646,"HVA4":698,"BLEND3_3":646},"ratings":{"BVS_CUSTOM":"C","HVA4":"B","BLEND3_3":"C"},"dt_nascimento":"1967-04-23","rendaConsideradaPessoa":4795,"rendaUtilizada":"SERASA","fatorCorrecaoRenda":1.37,"limiteCalculadoPessoa":2157.75,"qtdeRestricoes":0,"valorTotalRestricao":0,"situacao_cadastral_serasa":"REGULAR","existeErroSerasa":false,"existeErroPowerCurve":false,"existeErroBvsCustom":false,"errosTecnicos":[],"dadosAusentes":[],"restricoesPorModalidade":[],"anotacoesFinanceirasSerasa":{"Pefin":{"DataPrimeiraOcorrenciaPefin":null,"DataUltimaOcorrenciaPefin":null,"QuantidadeTotalPefin":"0","ValorTotalPefin":"0"},"Refin":{"DataPrimeiraOcorrenciaRefin":null,"DataUltimaOcorrenciaRefin":null,"QuantidadeTotalRefin":"0","ValorTotalRefin":"0"},"Acao":{"DataPrimeiraAcao":null,"DataUltimaAcao":null,"QuantidadeTotalAcao":null,"ValorTotalA

In [11]:
parsed = credpago_df["json_retornado"].apply(
    lambda x: json.loads(x) if pd.notna(x) and x else None
)

valid_parsed = parsed.dropna()

contrato_df = pd.json_normalize(valid_parsed, sep="_")  # index = idx original

pessoa_df = pd.json_normalize(
    valid_parsed.apply(lambda x: (x.get("message", {}).get("pessoas") or [{}])[0]),
    sep="_",
).add_prefix("pessoa_")

In [12]:
valid = parsed.notna()
base_idx = credpago_df.loc[valid].index  # índice original

# pendencias_df (como você já montou)
pendencias = []
for idx, row in parsed[valid].items():
    pessoas = row.get("message", {}).get("pessoas") or []
    for p in pessoas:
        for pend in p.get("anotacoesFinanceirasSerasa", {}).get("PendenciasPagamentoPF", []):
            pendencias.append({"idx": idx, **pend})

pendencias_df = pd.DataFrame(pendencias)

# Agregação por consulta
if not pendencias_df.empty:
    pendencias_agg = (
        pendencias_df
        .groupby("idx", as_index=False)
        .agg(
            qtde_pefin=("Valor", "count"),
            valor_pefin_total=("Valor", lambda s: pd.to_numeric(s, errors="coerce").sum()),
            modalidades_pefin=("Modalidade", lambda s: ",".join(sorted(set(s.dropna())))),
        )
    )
else:
    pendencias_agg = pd.DataFrame(columns=["idx", "qtde_pefin", "valor_pefin_total", "modalidades_pefin"])

In [13]:
credpago_expandido = (
    credpago_df.loc[valid]
    .join(contrato_df, how="left")   # contrato_df index = parsed.dropna().index
    .join(pessoa_df, how="left")
    .reset_index(names="idx")
    .merge(pendencias_agg, on="idx", how="left")
    .drop(columns="idx")
)

In [14]:
cols_drop = [
    # ingestão
    "_sdc_table_version", "_sdc_received_at", "_sdc_sequence", "_sdc_batched_at",

    # json bruto / containers
    "json_retornado",
    "message_pessoas", "message_scores", "message_ratings",
    "pessoa_scores", "pessoa_ratings", "message_blend3Variables",

    # bronze redundante (validado)
    "score_bvs",
    "renda_considerada",
    "decisao_bureau",
    "limite_mensal",          # não bate com JSON; usar message_limiteCalculadoContrato

    # prováveis duplicatas (confirmar com teste acima)
    "id_externo",
    "data",                   # se requested_at bastar

    # derivável
    "request_month",

    # operacional / baixo valor
    "success", "message_manual", "message_node",
    "message_reaproveitamentoConsultaMotor", "message_savingBureauPowerCurve",
    "message_logs", "message_partnerIds",
    "message_errosTecnicos", "message_dadosAusentes",
    "pessoa_errosTecnicos", "pessoa_dadosAusentes",
    "message_rentGuaranteeConstraints_rent_coverage_multiplier",
    "message_rentGuaranteeConstraints_exit_cost_multiplier",
    "message_rentGuaranteeConstraints_commission_percent",
    "message_rentGuaranteeConstraints_version",
]

credpago_clean = credpago_expandido.drop(columns=[c for c in cols_drop if c in credpago_expandido.columns])

In [15]:
credpago_clean

,CPF_CNPJ,contract_id,requested_at,id_consulta,result_pre_analise,id_bureau,risco_cartoes,id_pessoa,protestados_outros_estados,renda_mensal,...,pessoa_anotacoesFinanceirasSerasa_ProtestosPF,pessoa_anotacoesFinanceirasSerasa_DividasVencidasPF,pessoa_anotacoesFinanceirasSerasa_ChequesSemFundosPF,pessoa_scores_HVA3,pessoa_ratings_HVA3,pessoa_scores_BLEND_REGRESSAO_2026,pessoa_ratings_BLEND_REGRESSAO_2026,qtde_pefin,valor_pefin_total,modalidades_pefin
0,70807521248,4016844,2026-05-08,5769969,APROVAR,1,0,<NA>,N,0,...,[],[],[],NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1390353036,4050748,2026-05-25,5844344,APROVAR,1,0,<NA>,N,0,...,"[{'DataOcorrencia': '20250630', 'Cartorio': '9...",[],[],NaN,NaN,NaN,NaN,1.0,215.0,CRED CARTAO
2,7114053525,4052776,2026-05-14,5794702,APROVAR,1,0,<NA>,N,0,...,[],[],[],NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,43119138851,4068320,2026-05-06,5756141,APROVAR,1,0,<NA>,N,0,...,[],[],[],NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1152804995,4069507,2026-05-04,5746287,APROVAR,1,0,<NA>,N,0,...,[],[],[],NaN,NaN,NaN,NaN,3.0,4614.0,"CRED CARTAO,FINANCIAMENT,OUTRAS OPER"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86492,3814735030,4173058,2026-05-25,5848917,APROVAR,1,0,<NA>,N,0,...,[],[],[],NaN,NaN,NaN,NaN,NaN,NaN,NaN
86493,1471724972,4173104,2026-05-25,5848972,APROVAR,1,0,<NA>,N,0,...,[],[],[],NaN,NaN,NaN,NaN,NaN,NaN,NaN
86494,11225532930,4173137,2026-05-25,5849010,APROVAR,1,0,<NA>,N,0,...,[],[],[],NaN,NaN,NaN,NaN,NaN,NaN,NaN
86495,70383410142,4173377,2026-05-25,5849310,APROVAR,1,0,<NA>,N,0,...,[],[],[],NaN,NaN,NaN,NaN,NaN,NaN,NaN


### CredPago - Imóvel e Histórico de Valor

In [27]:
query = """
WITH property_type AS (
  SELECT
    id AS contract_id,
    CASE WHEN tp_imovel = 2 THEN 0 ELSE 1 END AS property_type,
    CAST(id_cidade_ibge AS INT64) AS id_cidade_ibge,    --nova


    TRIM(
    REGEXP_REPLACE(
        REGEXP_REPLACE(
        UPPER(
            REGEXP_REPLACE(
            NORMALIZE(COALESCE(cidade, ''), NFD),
            r'\pM', ''                      -- tira acento (marcas)
            )
        ),
        r'[^A-Z0-9 ]', ' '                 -- tira lixo (agora já está tudo em maiúsculo)
        ),
        r'\s+', ' '                          -- colapsa espaços
    )
    ) AS contract_city_nm, --nova

    CAST(id_imobiliaria AS INT64) AS agency_id,  --nova
    DATE(criado_em) AS requested_at
  FROM `loft-dl-fintech.bronze_credpago.imovel`
  WHERE DATE(criado_em) >= DATE('2026-01-01')
  QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY criado_em DESC) = 1
),

rental_value AS (
  SELECT
    id_imovel AS contract_id,
    CAST(vl_aluguel AS FLOAT64) AS rental_value,
    DATE(data) AS requested_at
  FROM `loft-dl-fintech.bronze_credpago.historico_valor`
  WHERE DATE(data) >= DATE('2026-01-01')
  QUALIFY ROW_NUMBER() OVER (PARTITION BY id_imovel ORDER BY data DESC) = 1
)

SELECT
  COALESCE(p.contract_id, r.contract_id) AS contract_id,
  GREATEST(p.requested_at, r.requested_at) AS requested_at,
  p.property_type,
  p.id_cidade_ibge,
  p.contract_city_nm,
  p.agency_id,
  r.rental_value
FROM property_type p
FULL OUTER JOIN rental_value r
  ON p.contract_id = r.contract_id;
"""

credpago_imovel_df = pd.read_gbq(query, project_id=project_id)

In [28]:
credpago_imovel_df.head()

,contract_id,requested_at,property_type,id_cidade_ibge,contract_city_nm,agency_id,rental_value
0,3569155,2026-01-02,1,3506508,BIRIGUI,3767,550.0
1,3569207,2026-01-02,1,4202107,BARRA VELHA,45205,1100.0
2,3569330,2026-01-02,1,3525904,JUNDIAI,25174,1300.0
3,3569407,2026-01-02,1,5201405,APARECIDA DE GOIANIA,8719,1800.0
4,3569731,2026-01-02,1,3538006,PINDAMONHANGABA,52291,1200.0


### Merge 

In [32]:
cp_final_df = pd.merge(credpago_clean, credpago_imovel_df.drop(columns=['requested_at']), on='contract_id', how='left')
cp_final_df.head()

,CPF_CNPJ,contract_id,requested_at,id_consulta,result_pre_analise,id_bureau,risco_cartoes,id_pessoa,protestados_outros_estados,renda_mensal,...,pessoa_scores_BLEND_REGRESSAO_2026,pessoa_ratings_BLEND_REGRESSAO_2026,qtde_pefin,valor_pefin_total,modalidades_pefin,property_type,id_cidade_ibge,contract_city_nm,agency_id,rental_value
0,70807521248,4016844,2026-05-08,5769969,APROVAR,1,0,<NA>,N,0,...,NaN,NaN,NaN,NaN,NaN,1,4101804,ARAUCARIA,36257,3000.0
1,1390353036,4050748,2026-05-25,5844344,APROVAR,1,0,<NA>,N,0,...,NaN,NaN,1.0,215.0,CRED CARTAO,1,4304606,CANOAS,1736,1150.0
2,7114053525,4052776,2026-05-14,5794702,APROVAR,1,0,<NA>,N,0,...,NaN,NaN,NaN,NaN,NaN,0,3548708,SAO BERNARDO DO CAMPO,19793,6000.0
3,43119138851,4068320,2026-05-06,5756141,APROVAR,1,0,<NA>,N,0,...,NaN,NaN,NaN,NaN,NaN,1,3525300,JAU,1669,1100.0
4,1152804995,4069507,2026-05-04,5746287,APROVAR,1,0,<NA>,N,0,...,NaN,NaN,3.0,4614.0,"CRED CARTAO,FINANCIAMENT,OUTRAS OPER",1,4104006,CAMPINA GRANDE DO SUL,11474,1540.0


In [33]:
cols_ = ["property_type", "rental_value", "id_cidade_ibge", "agency_id"]

pct_nulls = (
    cp_final_df[cols_].isna().mean().mul(100).sort_values(ascending=False)
)
pct_nulls

rental_value      0.030059
property_type     0.006937
id_cidade_ibge    0.006937
agency_id         0.006937
dtype: float64

In [34]:
pct_nulls_por_requested_at = (
    cp_final_df.groupby("requested_at")[cols_]
    .apply(lambda x: x.isna().mean().mul(100))
    .sort_index()
)

pct_nulls_por_requested_at

,property_type,rental_value,id_cidade_ibge,agency_id
requested_at,,,,
2026-05-01,0.000000,0.000000,0.000000,0.000000
2026-05-02,0.000000,0.000000,0.000000,0.000000
2026-05-03,0.000000,0.000000,0.000000,0.000000
2026-05-04,0.032180,0.032180,0.032180,0.032180
2026-05-05,0.017159,0.000000,0.017159,0.017159
2026-05-06,0.000000,0.000000,0.000000,0.000000
2026-05-07,0.000000,0.000000,0.000000,0.000000
2026-05-08,0.022810,0.022810,0.022810,0.022810
2026-05-09,0.000000,0.000000,0.000000,0.000000


## Variáveis do Blend 3

In [35]:
cp_final_df[[col for col in cp_final_df.columns if "blend3Variables" in col]]

,message_blend3Variables_hasPreviousContracts,message_blend3Variables_hadOverdueInvoiceInPreviousContracts,message_blend3Variables_cityPc4HistoryOver100Contracts,message_blend3Variables_realEstatePc4HistoryOver100Contracts
0,False,False,NaN,NaN
1,False,False,0.088235,NaN
2,False,False,0.050802,NaN
3,False,False,0.098166,0.098039
4,False,False,NaN,NaN
...,...,...,...,...
86492,False,False,0.120244,NaN
86493,False,False,0.095238,NaN
86494,False,False,0.133803,NaN
86495,False,False,0.160000,NaN


In [50]:
cp_final_df['message_blendRegressaoPredict']

0        339.0
1        405.0
2        654.0
3        749.0
4        432.0
         ...  
86492    579.0
86493    306.0
86494    426.0
86495    374.0
86496    539.0
Name: message_blendRegressaoPredict, Length: 86497, dtype: float64

In [53]:
rename_dict = {
    "message_scores_BVS_CUSTOM": "SCRCRDPNMGRLPFLGBCLFCREDPGV1",
    "message_scores_HVA4": "SERASA_HVA4",
    "pessoa_idade": "age",
    "property_type": "property_type",  # peguei de fora da consulta realizada
    "message_qtdeRestricoesContrato": "qtde_restricoes__consulta_realizada",
    "rental_value": "rental_value",  # peguei de fora da consulta realizada
    "message_rendaConsideradaContrato": "income",
    "message_blend3Variables_realEstatePc4HistoryOver100Contracts": "agency_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_cityPc4HistoryOver100Contracts": "city_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_hasPreviousContracts": "flag_tem__contratos_anteriores",
    "message_blend3Variables_hadOverdueInvoiceInPreviousContracts": "flag_teve_boleto_atrasado__contratos_anteriores",
}

## Criar Variáveis

In [56]:
cp_final_df.columns.to_list()

['CPF_CNPJ',
 'contract_id',
 'requested_at',
 'id_consulta',
 'result_pre_analise',
 'id_bureau',
 'risco_cartoes',
 'id_pessoa',
 'protestados_outros_estados',
 'renda_mensal',
 'ocorrencia_debito',
 'score_bureau',
 'descricao_probabilidade_inadimplencia',
 'motor',
 'protesto_sao_paulo',
 'probabilidade_inadimplencia',
 'score_serasa',
 'score_medio_4kst',
 'regua',
 'documento',
 'letra',
 'idade',
 'documento_novo',
 'faixa_renda',
 'score_imobiliaria',
 'score_hvar',
 'id_funcionalidade',
 'score_hva3',
 'message_regras_situacao_cadastral_ph3a',
 'message_regras_similaridade_nome_receita_ph3a',
 'message_regras_situacao_cadastral_serasa',
 'message_regras_similaridade_nome_receita_federal_pj',
 'message_rendaConsiderada',
 'message_rendaConsideradaContrato',
 'message_rendaUtilizada',
 'message_fatorCorrecaoRenda',
 'message_limiteCalculadoContrato',
 'message_qtdeRestricoesContrato',
 'message_restricoesPorModalidadeContrato',
 'message_regra_estouro_limite',
 'message_resultad

In [64]:
vars_blend = ['SCRCRDPNMGRLPFLGBCLFCREDPGV1', 'SERASA_HVA4', 'age', 'property_type', 'qtde_restricoes__consulta_realizada', 'rental_value', 'income', 'agency_pc4_mais_100_contratos__pc_categorias', 'city_pc4_mais_100_contratos__pc_categorias', 'flag_tem__contratos_anteriores', 'flag_teve_boleto_atrasado__contratos_anteriores']

id_vars = ['contract_id', 'requested_at', 'CPF_CNPJ', 'message_decisao', 'message_blendRegressaoPredict']

vars_df = (cp_final_df.copy()).rename(columns=rename_dict)
vars_df = vars_df[id_vars+vars_blend]
vars_df.head()

,contract_id,requested_at,CPF_CNPJ,message_decisao,message_blendRegressaoPredict,SCRCRDPNMGRLPFLGBCLFCREDPGV1,SERASA_HVA4,age,property_type,qtde_restricoes__consulta_realizada,rental_value,income,agency_pc4_mais_100_contratos__pc_categorias,city_pc4_mais_100_contratos__pc_categorias,flag_tem__contratos_anteriores,flag_teve_boleto_atrasado__contratos_anteriores
0,4016844,2026-05-08,70807521248,BLEND3_3,339.0,583.0,642.0,48.0,1,0,3000.0,1986.5,NaN,NaN,False,False
1,4050748,2026-05-25,1390353036,BLEND3_3,405.0,506.0,572.0,38.0,1,4,1150.0,2603.0,NaN,0.088235,False,False
2,4052776,2026-05-14,7114053525,BLEND3_3,654.0,792.0,689.0,29.0,0,0,6000.0,7261.0,NaN,0.050802,False,False
3,4068320,2026-05-06,43119138851,BLEND3_3,749.0,796.0,888.0,23.0,1,0,1100.0,4452.5,0.098039,0.098166,False,False
4,4069507,2026-05-04,1152804995,BLEND3_3,432.0,581.0,509.0,33.0,1,3,1540.0,3219.5,NaN,NaN,False,False


In [65]:
vars_df["income_commitment"] = (
    vars_df["rental_value"]
    / vars_df["income"]
)

vars_df["income_commitment"] = (
    vars_df["income_commitment"].replace(
        [np.inf, -np.inf], np.nan
    )
)

BVS_cust_median = 700
BVS_cust_iqr = 182
SERASA_HVA4_median = 668
SERASA_HVA4_iqr = 246
age_median = 38
age_iqr = 21
qtde_restricoes__consulta_realizada_median = 0
qtde_restricoes__consulta_realizada_iqr = 1
income_commitment_median = 0.33291792771942325  # mudou o valor com relação ao blend3
income_commitment_iqr = 0.35937397251265873  # mudou o valor com relação ao blend3
### novas variáveis internas:
agency_pc4_mais_100_contratos__pc_categorias_median = 0.1238938053097345
agency_pc4_mais_100_contratos__pc_categorias_iqr = 1
city_pc4_mais_100_contratos__pc_categorias_median = 0.10323574730354394
city_pc4_mais_100_contratos__pc_categorias_iqr = 0.03233543652391829

fill_dict1 = {
    "SCRCRDPNMGRLPFLGBCLFCREDPGV1": 1,
    "SERASA_HVA4": 1,
    "age": age_median,
    "qtde_restricoes__consulta_realizada": qtde_restricoes__consulta_realizada_median,
    "income_commitment": income_commitment_median,
    "property_type": 1,
    "agency_pc4_mais_100_contratos__pc_categorias": agency_pc4_mais_100_contratos__pc_categorias_median,
    "city_pc4_mais_100_contratos__pc_categorias": city_pc4_mais_100_contratos__pc_categorias_median,
    "flag_tem__contratos_anteriores": 0,
    "flag_teve_boleto_atrasado__contratos_anteriores": 0,
}

cols_flag = [
    "city_pc4_mais_100_contratos__pc_categorias",
    "agency_pc4_mais_100_contratos__pc_categorias",
]

for col in cols_flag:
    vars_df[f"{col}_is_null"] = (
        vars_df[col].isna().astype(int)
    )

In [66]:
df_predict = vars_df.fillna(
    fill_dict1
)

df_predict["SCRCRDPNMGRLPFLGBCLFCREDPGV1__normalized3_2"] = (
    df_predict["SCRCRDPNMGRLPFLGBCLFCREDPGV1"] - BVS_cust_median
) / BVS_cust_iqr
df_predict["SERASA_HVA4__normalized3_2"] = (
    df_predict["SERASA_HVA4"] - SERASA_HVA4_median
) / SERASA_HVA4_iqr
df_predict["age__normalized3_2"] = (df_predict["age"] - age_median) / age_iqr
df_predict["qtde_restricoes__consulta_realizada__normalized3_2"] = (
    df_predict["qtde_restricoes__consulta_realizada"]
    - qtde_restricoes__consulta_realizada_median
) / qtde_restricoes__consulta_realizada_iqr
df_predict["income_commitment__normalized3_2"] = (
    df_predict["income_commitment"] - income_commitment_median
) / income_commitment_iqr
# novas variáveis internas
df_predict["agency_pc4_mais_100_contratos__pc_categorias__normalized3_2"] = (
    df_predict["agency_pc4_mais_100_contratos__pc_categorias"]
    - agency_pc4_mais_100_contratos__pc_categorias_median
) / agency_pc4_mais_100_contratos__pc_categorias_iqr
df_predict["city_pc4_mais_100_contratos__pc_categorias__normalized3_2"] = (
    df_predict["city_pc4_mais_100_contratos__pc_categorias"]
    - city_pc4_mais_100_contratos__pc_categorias_median
) / city_pc4_mais_100_contratos__pc_categorias_iqr

In [ ]:
df_predict = df_predict.assign(
    predict_blend3_2=lambda x: (
        -0.127297036249813
        + (-0.297212783378539) * x["SCRCRDPNMGRLPFLGBCLFCREDPGV1__normalized3_2"]
        + (-0.408969122005144) * x["SERASA_HVA4__normalized3_2"]
        + (0.292732320546562) * x["age__normalized3_2"]
        + (-0.134202392009538) * x["property_type"]
        + (0.0890819567329526) * x["qtde_restricoes__consulta_realizada__normalized3_2"]
        + (0.235331129159451) * x["income_commitment__normalized3_2"]
        + (0.0855288113746228)
        * x["agency_pc4_mais_100_contratos__pc_categorias__normalized3_2"]
        + (0.228478398386125)
        * x["city_pc4_mais_100_contratos__pc_categorias__normalized3_2"]
        + (-0.0822948528344378) * x["flag_tem__contratos_anteriores"]
        + (0.151958223727146) * x["flag_teve_boleto_atrasado__contratos_anteriores"]
        + (-0.149942669861252)
        * x["agency_pc4_mais_100_contratos__pc_categorias_is_null"]
        + (-0.0655542165633087)
        * x["city_pc4_mais_100_contratos__pc_categorias_is_null"]
    )
).assign(predict_blend3_2=lambda x: 1 / (1 + np.exp(-x["predict_blend3_2"].astype(float))))

In [76]:
df_predict

,contract_id,requested_at,CPF_CNPJ,message_decisao,message_blendRegressaoPredict,SCRCRDPNMGRLPFLGBCLFCREDPGV1,SERASA_HVA4,age,property_type,qtde_restricoes__consulta_realizada,...,city_pc4_mais_100_contratos__pc_categorias_is_null,agency_pc4_mais_100_contratos__pc_categorias_is_null,SCRCRDPNMGRLPFLGBCLFCREDPGV1__normalized3_2,SERASA_HVA4__normalized3_2,age__normalized3_2,qtde_restricoes__consulta_realizada__normalized3_2,income_commitment__normalized3_2,agency_pc4_mais_100_contratos__pc_categorias__normalized3_2,city_pc4_mais_100_contratos__pc_categorias__normalized3_2,predict_blend3_2
0,4016844,2026-05-08,70807521248,BLEND3_3,339.0,583.0,642.0,48.0,1,0,...,1,1,-0.642857,-0.105691,0.476190,0.0,3.275907,0.000000,0.000000,1.000000e+00
1,4050748,2026-05-25,1390353036,BLEND3_3,405.0,506.0,572.0,38.0,1,4,...,0,1,-1.065934,-0.390244,0.000000,4.0,0.302971,0.000000,-0.463901,1.000000e+00
2,4052776,2026-05-14,7114053525,BLEND3_3,654.0,792.0,689.0,29.0,0,0,...,0,1,0.505495,0.085366,-0.428571,0.0,1.372983,0.000000,-1.621552,1.325363e-276
3,4068320,2026-05-06,43119138851,BLEND3_3,749.0,796.0,888.0,23.0,1,0,...,0,0,0.527473,0.894309,-0.714286,0.0,-0.238931,-0.025855,-0.156782,0.000000e+00
4,4069507,2026-05-04,1152804995,BLEND3_3,432.0,581.0,509.0,33.0,1,3,...,1,1,-0.653846,-0.646341,-0.238095,3.0,0.404640,0.000000,0.000000,1.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86492,4173058,2026-05-25,3814735030,BLEND3_3,579.0,674.0,589.0,28.0,1,0,...,0,1,-0.142857,-0.321138,-0.476190,0.0,-0.260446,0.000000,0.525980,6.681602e-139
86493,4173104,2026-05-25,1471724972,BLEND3_3,306.0,439.0,215.0,31.0,1,0,...,0,1,-1.434066,-1.841463,-0.333333,0.0,0.879046,0.000000,-0.247334,1.000000e+00
86494,4173137,2026-05-25,11225532930,BLEND3_3,426.0,491.0,610.0,27.0,0,0,...,0,1,-1.148352,-0.235772,-0.523810,0.0,0.324661,0.000000,0.945312,1.000000e+00
86495,4173377,2026-05-25,70383410142,BLEND3_3,374.0,524.0,560.0,27.0,1,0,...,0,1,-0.967033,-0.439024,-0.523810,0.0,0.894610,0.000000,1.755481,1.000000e+00


In [80]:
df_predict['1-predict_blend3_2'] = np.round((1-df_predict['predict_blend3_2'])*1000,0)
df_predict[['predict_blend3_2', 'message_blendRegressaoPredict', '1-predict_blend3_2']].head()

,predict_blend3_2,message_blendRegressaoPredict,1-predict_blend3_2
0,0.660968,339.0,339.0
1,0.595464,405.0,405.0
2,0.346325,654.0,654.0
3,0.252114,749.0,748.0
4,0.568183,432.0,432.0


### Credit Fact

In [19]:
query_cf = '''
WITH 
SIMULATIONS AS (
  SELECT * 
  FROM `loft-dl-fintech.cp_gold.credit_fact` AS CF
  WHERE DATE(requested_at) >= '2026-05-01'
)
SELECT *
FROM SIMULATIONS
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY contract_id
    ORDER BY requested_at DESC
) = 1
'''
creditfact_df = pd.read_gbq(query_cf, project_id=project_id)
creditfact_df

,contract_id,requested_at,agency_id,score_imobiliaria,risco_imobiliario,tipo_contrato,faixas_rental_value,regiao,contract_city_nm,contract_state_nm,...,tipo,faixa_resticao,decisao_PJ,Imob_risco3M,ajuste_poliica,valor,politica,modeloBlend,blendRegressaoPredict,score_BVS_CUSTOM
0,4083424,2026-05-01 00:14:58+00:00,56116,NI,7.NI,PF,1. Ate 3 mil,Sudeste,Bauru,SP,...,PF,Acima de 5K,DERIVAR,Sem risco,Referência,1. Ate 3 mil,37,BLEND3_3,382.0,718.000000000
1,4083425,2026-05-01 01:11:08+00:00,56116,NI,7.NI,PF,1. Ate 3 mil,Sudeste,Bauru,SP,...,PF,De 1 a 1K,APROVAR,Sem risco,Referência,1. Ate 3 mil,33,BLEND3_3,333.0,548.000000000
2,4083426,2026-05-01 03:40:18+00:00,16060,NI,7.NI,PF,2. De 3 a 5 mil,Sudeste,São José dos Campos,SP,...,PF,sem restrição,REPROVAR,Sem risco,Referência,2. De 3 a 5 mil,33,BLEND3_3,NaN,274.000000000
3,4083427,2026-05-01 04:03:45+00:00,11294,NI,7.NI,PF,1. Ate 3 mil,Sudeste,Cabo Frio,RJ,...,PF,sem restrição,APROVAR,Sem risco,Referência,1. Ate 3 mil,34,BLEND3_3,433.0,661.000000000
4,4083428,2026-05-01 04:07:56+00:00,11294,NI,7.NI,PF,1. Ate 3 mil,Sudeste,Cabo Frio,RJ,...,PF,De 1k a 5K,APROVAR,Sem risco,Referência,1. Ate 3 mil,31,BLEND3_3,460.0,848.000000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89500,4173376,2026-05-25 18:24:00+00:00,54787,None,8.Não classificado na base,PF,1. Ate 3 mil,Sul,Bento Gonçalves,RS,...,PF,sem restrição,DERIVAR,Sem risco,Referência,1. Ate 3 mil,None,None,NaN,None
89501,4173377,2026-05-25 18:31:18+00:00,21966,None,8.Não classificado na base,PF,1. Ate 3 mil,Centro-Oeste,Goiânia,GO,...,PF,sem restrição,DERIVAR,Sem risco,Referência,1. Ate 3 mil,None,None,NaN,None
89502,4173378,2026-05-25 18:33:03+00:00,28596,None,8.Não classificado na base,PF,1. Ate 3 mil,Sudeste,Pariquera-Açu,SP,...,PF,sem restrição,DERIVAR,Sem risco,Referência,1. Ate 3 mil,None,None,NaN,None
89503,4173379,2026-05-25 18:39:56+00:00,41690,None,8.Não classificado na base,PF,1. Ate 3 mil,Norte - Nordeste,Caucaia,CE,...,PF,sem restrição,DERIVAR,Sem risco,Referência,1. Ate 3 mil,None,None,NaN,None
